# Non-exogenous regressors: why the robust variance matters

With a regressor that responds to past shocks, the naive GLS variance ([PG26] eq. (27)) ignores the estimation of the AR filter; the robust variance (eqs. (28)–(29)) does not.

In [1]:
import warnings

import numpy as np
import pandas as pd

import pyfgls as pf

warnings.simplefilter("ignore", pf.exceptions.PyFGLSWarning)


In [2]:
from pyfgls.simulate import serial_dgp

sim = serial_dgp(200, ar=(0.5,), rho_x=0.8, gamma=1.0, burn=100, random_state=3)
robust = pf.SerialFGLS(sim.y, sim.X).fit()
naive = pf.SerialFGLS(sim.y, sim.X).fit(cov_type='naive')
print('nonexogeneity share:', round(robust.nonexogeneity_share, 3))
pf.compare([naive, robust]).round(4)

nonexogeneity share: 0.463


FGLS (serial) [naive]  FGLS (serial) [robust]
const coef                     1.2222                  1.2222
      std err                  0.1488                  0.1570
      CI width                 0.5834                  0.6153
x1    coef                     0.9930                  0.9930
      std err                  0.0472                  0.0636
      CI width                 0.1852                  0.2495

Coverage of the two 95% intervals in a small simulation:

In [3]:
from pyfgls.serial import estimate_serial

rng = np.random.default_rng(4)
cov_r = cov_n = 0
R = 400
for _ in range(R):
    s = serial_dgp(200, ar=(0.5,), rho_x=0.8, gamma=1.0, burn=100, random_state=rng)
    e = estimate_serial(s.y, s.X, ['c', 'x'], kmax=12)
    b = e.beta[1]
    cov_r += abs(b - 1) < 1.96 * np.sqrt(e.cov[1, 1])
    cov_n += abs(b - 1) < 1.96 * np.sqrt(e.variance.naive_avar[1, 1] / e.variance.n)
print(f'robust: {cov_r / R:.3f}   naive: {cov_n / R:.3f}')

robust: 0.953   naive: 0.858


## Lagged dependent variables

Declare them; the AR filter is then recovered by the Wallis ratio ([PG26] Section 4.2).

In [4]:
from pyfgls.simulate import arma_errors

rng = np.random.default_rng(5)
T = 500
w = rng.normal(size=T + 1)
u = arma_errors(rng.normal(size=T + 1), (0.5,))
yy = np.zeros(T + 1)
for t in range(1, T + 1):
    yy[t] = 0.5 * yy[t - 1] + 1.0 + w[t] + u[t]
Xd = pd.DataFrame({'const': 1.0, 'ylag': yy[:-1], 'w': w[1:]})
dyn = pf.SerialFGLS(yy[1:], Xd, lagged_dep_vars=['ylag']).fit()
dyn.compare_ols().round(3)

OLS  FGLS (serial)
const coef      0.609          1.104
      std err   0.093          0.158
      CI width  0.364          0.620
ylag  coef      0.699          0.454
      std err   0.026          0.054
      CI width  0.102          0.213
w     coef      1.028          1.042
      std err   0.050          0.048
      CI width  0.195          0.190

## Is the model dynamically complete? (experimental)

[PG26] Remark 9: test whether lags of the regressors are missing.

In [5]:
from pyfgls.diagnostics import test_predetermined

test_predetermined(sim.y, sim.X, max_lag=3)

Pre-determinedness LM test ([PG26] Remark 9, experimental)
Durbin AR order: 1; regressors tested: x1
         LM  df  p-value
lags                    
1    0.1230   1   0.7258
2    0.1778   2   0.9150
3    0.3554   3   0.9493
No rejection at 5%: no evidence of missing regressor lags.

Now a model that omits $x_{t-1}$: the true equation is $y_t = 1 + x_t + 0.4x_{t-1} + u_t$, so $x_t$ is correlated with the composite error. The test rejects, and adding the lag fixes it.

In [6]:
rng = np.random.default_rng(7)
x = rng.normal(size=401)
u = arma_errors(rng.normal(size=400), (0.5,))
y2 = 1.0 + x[1:] + 0.4 * x[:-1] + u
X_short = pd.DataFrame({'const': 1.0, 'x': x[1:]})
print(test_predetermined(y2, X_short, max_lag=2))
X_long = X_short.assign(x_lag1=x[:-1])
print(test_predetermined(y2, X_long, max_lag=2, regressors=['x']))
pf.SerialFGLS(y2, X_long).fit().params.round(3)

Pre-determinedness LM test ([PG26] Remark 9, experimental)
Durbin AR order: 2; regressors tested: x
          LM  df  p-value
lags                     
1    41.1311   1   0.0000
2    41.2666   2   0.0000
Rejected at 5% with 1 lag(s): the regressors may not be pre-determined. Add lag(s) 1..1 of x as regressors, then refit.
Pre-determinedness LM test ([PG26] Remark 9, experimental)
Durbin AR order: 1; regressors tested: x
         LM  df  p-value
lags                    
1    0.0000   0   1.0000
2    0.1670   1   0.6828
No rejection at 5%: no evidence of missing regressor lags.


const     0.776
x         0.947
x_lag1    0.346
Name: params, dtype: float64